<a href="https://colab.research.google.com/github/shaan-byte/python_ds_colab/blob/main/Advanced_Data_Transformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 6.5 — Advanced Data Transformation

**Module 2: Data Analysis Foundations**  
**Week 6 | Session 6.5**

---

## What we will cover today

1. Apply functions — running custom logic across every row or column
2. Map and Replace — translating values from one representation to another
3. Pivot and Melt — reshaping data between wide and long format
4. Date/Time handling — extracting meaning from timestamps

**Session duration:** 1.5 hours  
**Format:** Live coding in Google Colab + virtual whiteboard

---

> **Where we left off:** We can load, inspect, filter, summarise, and join data. Today we handle the transformation step — taking raw data and reshaping or recoding it into something more useful for analysis. We continue with the employee dataset for apply/map/pivot, then introduce a sales transactions dataset for DateTime where richer time-series data lets us demonstrate the full toolkit.

---
## Setup — Run this first

In [ ]:
datetime(2023, 1, 1)

datetime.datetime(2023, 1, 1, 0, 0)

In [ ]:
import pandas as pd
import numpy as np
import csv, random
from datetime import datetime, timedelta

# ── Employee dataset (same as previous sessions) ──────────────────────────────
random.seed(42)
departments_list = ["Engineering", "Sales", "HR", "Finance"]
cities      = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad"]
first_names = ["Ananya","Rohit","Priya","Vikram","Sneha","Arjun",
               "Deepa","Kiran","Meera","Suresh","Lakshmi","Rahul",
               "Divya","Arun","Pooja","Sanjay","Nisha","Tarun"]
last_names  = ["Sharma","Mehta","Nair","Rao","Pillai","Das",
               "Joshi","Iyer","Bose","Kumar","Reddy","Singh"]
dept_salary = {"Engineering":(70000,150000),"Sales":(55000,110000),
               "HR":(50000,95000),"Finance":(65000,130000)}
rows = []
for i in range(50):
    dept   = random.choice(departments_list)
    lo, hi = dept_salary[dept]
    salary = round(random.uniform(lo, hi), -3)
    perf   = round(random.uniform(2.0, 5.0), 1)
    year   = random.randint(2004, 2023)
    month  = random.randint(1, 12)
    row    = [1001+i,
              random.choice(first_names)+" "+random.choice(last_names),
              dept if random.random()>=0.06 else dept.lower(),
              "" if random.random()<0.08 else salary,
              random.randint(1,20),
              random.choice(cities),
              "" if random.random()<0.08 else perf,
              f"{year}-{month:02d}-01",
              random.choice([True,True,True,False])]
    rows.append(row)
with open("employees.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["employee_id","name","department","salary",
                "experience","city","performance_rating","join_date","active"])
    w.writerows(rows)

emp = pd.read_csv("employees.csv", parse_dates=["join_date"])
emp["salary"]             = pd.to_numeric(emp["salary"],             errors="coerce")
emp["performance_rating"] = pd.to_numeric(emp["performance_rating"], errors="coerce")
emp["department"]         = emp["department"].str.title()

# ── Sales transactions dataset (used in Part 4 — DateTime) ───────────────────
random.seed(7)
products   = ["Laptop", "Monitor", "Keyboard", "Mouse", "Headset", "Webcam", "Docking Station"]
categories = {"Laptop":"Computing", "Monitor":"Displays", "Keyboard":"Peripherals",
              "Mouse":"Peripherals", "Headset":"Audio", "Webcam":"Video", "Docking Station":"Computing"}
regions    = ["North", "South", "East", "West"]
base_price = {"Laptop":65000, "Monitor":22000, "Keyboard":3500, "Mouse":1200,
              "Headset":4500, "Webcam":5000, "Docking Station":12000}

base_date  = datetime(2023, 1, 1)
tx_rows    = []
for i in range(300):
    product  = random.choice(products)
    region   = random.choice(regions)
    qty      = random.randint(1, 5)
    price    = round(base_price[product] * random.uniform(0.85, 1.15), -2)
    days_off = random.randint(0, 364)
    hours    = random.randint(8, 20)
    minutes  = random.randint(0, 59)
    tx_date  = base_date + timedelta(days=days_off, hours=hours, minutes=minutes)
    tx_rows.append([f"TX{i+1:04d}", tx_date, product, categories[product],
                    region, qty, price, qty * price])

sales = pd.DataFrame(tx_rows,
    columns=["transaction_id","timestamp","product","category",
             "region","quantity","unit_price","total_amount"])
sales["timestamp"] = pd.to_datetime(sales["timestamp"])

print("Employee dataset:", emp.shape)
print("Sales dataset:   ", sales.shape)
print()
print(emp.head(2))
print()
print(sales.head(2))

Employee dataset: (50, 9)
Sales dataset:    (300, 8)

   employee_id          name   department   salary  experience     city  \
0         1001  Vikram Reddy  Engineering  72000.0          19  Chennai   
1         1002   Tarun Joshi        Sales  68000.0           1    Delhi   

   performance_rating  join_date  active  
0                 NaN 2011-03-01    True  
1                 3.8 2021-04-01    True  

  transaction_id           timestamp   product     category region  quantity  \
0         TX0001 2023-02-07 16:06:00  Keyboard  Peripherals  South         4   
1         TX0002 2023-02-14 14:26:00  Keyboard  Peripherals  North         5   

   unit_price  total_amount  
0      3700.0       14800.0  
1      3200.0       16000.0  


In [ ]:
emp

,employee_id,name,department,salary,experience,city,performance_rating,join_date,active
0,1001,Vikram Reddy,Engineering,72000.0,19,Chennai,NaN,2011-03-01,True
1,1002,Tarun Joshi,Sales,68000.0,1,Delhi,3.8,2021-04-01,True
2,1003,Priya Joshi,Hr,57000.0,20,Bangalore,4.9,2014-02-01,False
3,1004,Rahul Kumar,Engineering,NaN,8,Bangalore,3.1,2021-05-01,True
4,1005,Rahul Das,Engineering,100000.0,3,Hyderabad,3.4,2015-03-01,True
5,1006,Lakshmi Sharma,Sales,NaN,11,Chennai,2.8,2021-04-01,True
6,1007,Meera Nair,Hr,60000.0,9,Hyderabad,3.5,2018-03-01,False
7,1008,Priya Sharma,Hr,60000.0,6,Chennai,5.0,2020-08-01,False
8,1009,Ananya Reddy,Finance,104000.0,18,Bangalore,3.4,2012-09-01,True
9,1010,Meera Bose,Engineering,93000.0,4,Bangalore,2.5,2004-12-01,True


In [ ]:
sales

,transaction_id,timestamp,product,category,region,quantity,unit_price,total_amount
0,TX0001,2023-02-07 16:06:00,Keyboard,Peripherals,South,4,3700.0,14800.0
1,TX0002,2023-02-14 14:26:00,Keyboard,Peripherals,North,5,3200.0,16000.0
2,TX0003,2023-01-31 17:07:00,Laptop,Computing,South,1,66000.0,66000.0
3,TX0004,2023-01-26 11:02:00,Monitor,Displays,North,5,22600.0,113000.0
4,TX0005,2023-10-04 09:36:00,Headset,Audio,South,3,4400.0,13200.0
...,...,...,...,...,...,...,...,...
295,TX0296,2023-01-31 20:32:00,Docking Station,Computing,West,5,13000.0,65000.0
296,TX0297,2023-12-11 19:44:00,Headset,Audio,West,5,4000.0,20000.0
297,TX0298,2023-11-21 15:40:00,Headset,Audio,North,2,3900.0,7800.0
298,TX0299,2023-01-19 14:49:00,Docking Station,Computing,South,1,12600.0,12600.0


---
# Part 1 — Apply Functions

## The problem that `.apply()` solves

Pandas' built-in methods (`.mean()`, `.str.title()`, arithmetic operators) cover most transformations. But sometimes your logic is too custom to express as a single method — you need to run your own Python function across every row or every column.

`.apply()` is the bridge between Pandas and arbitrary Python logic.

```
df['col'].apply(func)   — runs func on every VALUE in a column    (Series)
df.apply(func, axis=1)  — runs func on every ROW                  (DataFrame row as a Series)
df.apply(func, axis=0)  — runs func on every COLUMN               (less common)
```

## An important note on performance

`.apply()` loops through rows in Python — it is significantly slower than vectorized operations. Use it **only when no vectorized alternative exists**. If you can achieve the same result with `.str` methods, arithmetic, or `np.where()`, do that instead.

In [ ]:
emp.head(8)

,employee_id,name,department,salary,experience,city,performance_rating,join_date,active,salary_band
0,1001,Vikram Reddy,Engineering,72000.0,19,Chennai,NaN,2011-03-01,True,Mid
1,1002,Tarun Joshi,Sales,68000.0,1,Delhi,3.8,2021-04-01,True,Mid
2,1003,Priya Joshi,Hr,57000.0,20,Bangalore,4.9,2014-02-01,False,Entry
3,1004,Rahul Kumar,Engineering,NaN,8,Bangalore,3.1,2021-05-01,True,Unknown
4,1005,Rahul Das,Engineering,100000.0,3,Hyderabad,3.4,2015-03-01,True,Senior
5,1006,Lakshmi Sharma,Sales,NaN,11,Chennai,2.8,2021-04-01,True,Unknown
6,1007,Meera Nair,Hr,60000.0,9,Hyderabad,3.5,2018-03-01,False,Mid
7,1008,Priya Sharma,Hr,60000.0,6,Chennai,5.0,2020-08-01,False,Mid


In [ ]:
# Applying a function to a Series (one column)
# Use case: classify salary into a band — logic too complex for a single expression

def salary_band(salary):
    """Converts a numeric salary into a descriptive band label."""
    if pd.isna(salary):
        return "Unknown"
    elif salary < 60000:
        return "Entry"
    elif salary < 90000:
        return "Mid"
    elif salary < 120000:
        return "Senior"
    else:
        return "Lead"

In [ ]:
emp["salary_band"] = emp["salary"].apply(salary_band)

In [ ]:
emp

,employee_id,name,department,salary,experience,city,performance_rating,join_date,active,salary_band
0,1001,Vikram Reddy,Engineering,72000.0,19,Chennai,NaN,2011-03-01,True,Mid
1,1002,Tarun Joshi,Sales,68000.0,1,Delhi,3.8,2021-04-01,True,Mid
2,1003,Priya Joshi,Hr,57000.0,20,Bangalore,4.9,2014-02-01,False,Entry
3,1004,Rahul Kumar,Engineering,NaN,8,Bangalore,3.1,2021-05-01,True,Unknown
4,1005,Rahul Das,Engineering,100000.0,3,Hyderabad,3.4,2015-03-01,True,Senior
5,1006,Lakshmi Sharma,Sales,NaN,11,Chennai,2.8,2021-04-01,True,Unknown
6,1007,Meera Nair,Hr,60000.0,9,Hyderabad,3.5,2018-03-01,False,Mid
7,1008,Priya Sharma,Hr,60000.0,6,Chennai,5.0,2020-08-01,False,Mid
8,1009,Ananya Reddy,Finance,104000.0,18,Bangalore,3.4,2012-09-01,True,Senior
9,1010,Meera Bose,Engineering,93000.0,4,Bangalore,2.5,2004-12-01,True,Senior


In [ ]:
def f(a):
  print(a)

In [ ]:
emp["employee_id"].apply(f)

1001
1002
1003
1004
1005
1006
1007
1008
1009
1010
1011
1012
1013
1014
1015
1016
1017
1018
1019
1020
1021
1022
1023
1024
1025
1026
1027
1028
1029
1030
1031
1032
1033
1034
1035
1036
1037
1038
1039
1040
1041
1042
1043
1044
1045
1046
1047
1048
1049
1050


,employee_id
0,None
1,None
2,None
3,None
4,None
5,None
6,None
7,None
8,None
9,None


In [ ]:
def func_name(param):
  if param > 10:
    return 10000
  return 20

In [ ]:
func_name(100)

10000

In [ ]:
l = lambda param: 10000

In [ ]:
l(20333)

10000

In [ ]:
# Lambda functions — one-liner anonymous functions, useful for simple transforms
# Write a lambda when the logic is short enough to fit on one line

# Format salary as a currency string — not achievable with a single built-in method
emp["salary_formatted"] = emp["salary"].apply(lambda x: f"₹{x:,.0f}" if pd.notna(x) else "N/A")

print("Salary formatted as currency string:")
print(emp[["name", "salary", "salary_formatted"]].head(8))

Salary formatted as currency string:
             name    salary salary_formatted
0    Vikram Reddy   72000.0          ₹72,000
1     Tarun Joshi   68000.0          ₹68,000
2     Priya Joshi   57000.0          ₹57,000
3     Rahul Kumar       NaN              N/A
4       Rahul Das  100000.0         ₹100,000
5  Lakshmi Sharma       NaN              N/A
6      Meera Nair   60000.0          ₹60,000
7    Priya Sharma   60000.0          ₹60,000


In [ ]:
# Row-wise apply — axis=1
# Each call receives a whole ROW as a Series, accessed by column name
# Use case: a composite score that combines MULTIPLE columns with custom weights

def composite_score(row):
    """
    Computes a composite employee score from salary, experience, and performance.
    Each component is weighted differently.
    Returns NaN if any input is missing.
    """
    if pd.isna(row["salary"]) or pd.isna(row["performance_rating"]):
        return np.nan

    # Normalise each component to a 0-100 scale, then weight
    salary_score  = min(row["salary"] / 150000 * 100, 100)   # cap at 100
    exp_score     = min(row["experience"] / 20 * 100, 100)    # 20 yrs = max
    perf_score    = (row["performance_rating"] / 5) * 100

    return round(salary_score * 0.4 + exp_score * 0.3 + perf_score * 0.3, 1)

emp["composite_score"] = emp.apply(composite_score, axis=1)

print("Composite employee scores:")
print(emp[["name", "salary", "experience", "performance_rating", "composite_score"]].head(8))
print()
top5 = emp[["name", "composite_score"]].dropna().sort_values("composite_score", ascending=False).head(5)
print("Top 5 by composite score:")
print(top5)

Composite employee scores:
             name    salary  experience  performance_rating  composite_score
0    Vikram Reddy   72000.0          19                 NaN              NaN
1     Tarun Joshi   68000.0           1                 3.8             42.4
2     Priya Joshi   57000.0          20                 4.9             74.6
3     Rahul Kumar       NaN           8                 3.1              NaN
4       Rahul Das  100000.0           3                 3.4             51.6
5  Lakshmi Sharma       NaN          11                 2.8              NaN
6      Meera Nair   60000.0           9                 3.5             50.5
7    Priya Sharma   60000.0           6                 5.0             55.0

Top 5 by composite score:
            name  composite_score
44     Sneha Rao             92.0
46  Suresh Reddy             86.7
30      Arun Das             83.2
22     Priya Rao             78.1
37    Tarun Iyer             77.2


In [ ]:
def sum(col):
  return col.sum()

In [ ]:
new_emp = emp[['salary', 'experience']].dropna()

In [ ]:
new_emp.apply(sum, axis=0)

,0
salary,4208000.0
experience,582.0


In [ ]:
# When NOT to use .apply() — show the vectorized alternative for the same task

# BAD: using .apply() for something built-in methods already handle
slow_result = emp["salary"].apply(lambda x: x * 1.10)   # 10% raise via apply

# GOOD: vectorized — same result, much faster on large data
fast_result = emp["salary"] * 1.10

print("Both produce the same values (first 5):")
print("apply:      ", slow_result.head(5).values)
print("vectorized: ", fast_result.head(5).values)
print()
print("Rule of thumb:")
print("  Can you write it as arithmetic / .str method / np.where?  -> use that")
print("  Do you need multi-column logic or a Python if/else tree?  -> use .apply()")

Both produce the same values (first 5):
apply:       [ 79200.  74800.  62700.     nan 110000.]
vectorized:  [ 79200.  74800.  62700.     nan 110000.]

Rule of thumb:
  Can you write it as arithmetic / .str method / np.where?  -> use that
  Do you need multi-column logic or a Python if/else tree?  -> use .apply()


---
### Exercise 1 — Apply Functions

**Task:** Complete the function and lambda below.

In [ ]:
# Q1: Apply a function that returns 'Long-tenure' if experience > 10, else 'Short-tenure'
def tenure_label(experience):
    if experience > 10:
        return "Long-tenure"
    return"Short-tenure"

emp["tenure_label"] = emp["experience"].apply(tenure_label)
print("Q1 Tenure labels:")
print(emp["tenure_label"].value_counts())

# Q2: Using a lambda, create a 'name_length' column with the number of characters in each name
emp["name_length"] = emp["name"].apply(lambda x: ___)
print(f"\nQ2 Name lengths (first 5): {emp['name_length'].head(5).tolist()}")

# Q3: Row-wise apply — create a 'reward_eligible' column that is True only if:
#     performance_rating >= 4.0 AND experience >= 5 AND active == True
def reward_eligible(row):
    if pd.isna(row["performance_rating"]):
        return False
    return (
        row["perfomance_rating"] >= 4.0 and
        row['experience'] >= 5   and
        row['active'] == True
    )

emp["reward_eligible"] = emp.apply(reward_eligible, axis=1)
print(f"\nQ3 Reward-eligible employees: {emp['reward_eligible'].sum()}")
print(emp[emp["reward_eligible"]][["name", "performance_rating", "experience", "active"]].head(6))

print()
print(f"Correct Q1 (2 categories):           {emp['tenure_label'].nunique() == 2}")
print(f"Correct Q2 (lengths > 0):            {(emp['name_length'] > 0).all()}")
print(f"Correct Q3 (all meet all 3 criteria):",
      (emp[emp['reward_eligible']][['performance_rating','experience']]
       .dropna().apply(lambda r: r['performance_rating']>=4.0 and r['experience']>=5, axis=1).all()))

<details>
<summary>Show solution</summary>

```python
def tenure_label(experience):
    if experience > 10:
        return "Long-tenure"
    return "Short-tenure"

emp["tenure_label"]    = emp["experience"].apply(tenure_label)
emp["name_length"]     = emp["name"].apply(lambda x: len(x))

def reward_eligible(row):
    if pd.isna(row["performance_rating"]):
        return False
    return (
        row["performance_rating"] >= 4.0 and
        row["experience"] >= 5 and
        row["active"] == True
    )

emp["reward_eligible"] = emp.apply(reward_eligible, axis=1)
```

</details>

---
# Part 2 — Map and Replace

## The idea

`.map()` and `.replace()` both translate values — they look up a value and swap it with something else. They are simpler and faster than `.apply()` for this specific task.

| Tool | Best for | Works on |
|---|---|---|
| `.map(dict)` | Translating every value using a lookup dictionary | Series only |
| `.replace(old, new)` | Substituting specific values (including lists) | Series and DataFrame |
| `.apply(func)` | Arbitrary custom logic per value / row | Series and DataFrame |

**Rule of thumb:** if your transformation is a straight lookup ("Sales" → "Revenue Team"), use `.map()` or `.replace()`. Only reach for `.apply()` when you need conditional logic.

In [ ]:
emp.head(8)

,employee_id,name,department,salary,experience,city,performance_rating,join_date,active,salary_band,salary_formatted,composite_score
0,1001,Vikram Reddy,Engineering,72000.0,19,Chennai,NaN,2011-03-01,True,Mid,"₹72,000",NaN
1,1002,Tarun Joshi,Sales,68000.0,1,Delhi,3.8,2021-04-01,True,Mid,"₹68,000",42.4
2,1003,Priya Joshi,Hr,57000.0,20,Bangalore,4.9,2014-02-01,False,Entry,"₹57,000",74.6
3,1004,Rahul Kumar,Engineering,NaN,8,Bangalore,3.1,2021-05-01,True,Unknown,N/A,NaN
4,1005,Rahul Das,Engineering,100000.0,3,Hyderabad,3.4,2015-03-01,True,Senior,"₹100,000",51.6
5,1006,Lakshmi Sharma,Sales,NaN,11,Chennai,2.8,2021-04-01,True,Unknown,N/A,NaN
6,1007,Meera Nair,Hr,60000.0,9,Hyderabad,3.5,2018-03-01,False,Mid,"₹60,000",50.5
7,1008,Priya Sharma,Hr,60000.0,6,Chennai,5.0,2020-08-01,False,Mid,"₹60,000",55.0


In [ ]:
# .map() with a dictionary — the cleanest lookup pattern
# Use case: translating department codes to full names, city to region, etc.

city_to_region = {
    "Mumbai":    "West",
    "Delhi":     "North",
    "Bangalore": "South",
    "Hyderabad": "South"
}

emp["region"] = emp["city"].replace(city_to_region)

print("City mapped to region:")
print(emp[["name", "city", "region"]].head(8))
print()
print("Region distribution:")
print(emp["region"].value_counts())

City mapped to region:
             name       city   region
0    Vikram Reddy    Chennai  Chennai
1     Tarun Joshi      Delhi    North
2     Priya Joshi  Bangalore    South
3     Rahul Kumar  Bangalore    South
4       Rahul Das  Hyderabad    South
5  Lakshmi Sharma    Chennai  Chennai
6      Meera Nair  Hyderabad    South
7    Priya Sharma    Chennai  Chennai

Region distribution:
region
South      17
Chennai    13
North      12
West        8
Name: count, dtype: int64


In [ ]:
# What happens when .map() encounters a value NOT in the dictionary?
# It returns NaN — not an error, but a silent gap

incomplete_map = {"Mumbai": "West", "Delhi": "North"}   # only two cities covered

test = emp["city"].map(incomplete_map)
print("Mapping with only 2 cities defined:")
print(test.value_counts(dropna=False))
print()
print("Cities NOT in the dict became NaN.")
print("Always check .isnull().sum() after .map() to catch missing mappings.")
print(f"NaN count: {test.isnull().sum()}")

Mapping with only 2 cities defined:
city
NaN      30
North    12
West      8
Name: count, dtype: int64

Cities NOT in the dict became NaN.
Always check .isnull().sum() after .map() to catch missing mappings.
NaN count: 30


In [ ]:
# .replace() — more flexible than .map()
# Key difference from .map(): .replace() ONLY changes specified values,
# leaving everything else as-is. .map() maps EVERY value (unspecified ones become NaN).

# Use case 1: fix a specific bad value
emp_copy = emp.copy()
emp_copy["active"] = emp_copy["active"].replace({True: "Yes", False: "No"})
print("Boolean replaced with Yes/No:")
print(emp_copy["active"].value_counts())
print()

# Use case 2: replace a list of bad values with a single fix
dept_fix = emp.copy()
dept_fix["department"] = dept_fix["department"].replace(
    ["engineering", "ENGINEERING", "Engg"], "Engineering"
)
print("Replacing variant spellings:")
print(dept_fix["department"].value_counts())
print()

# Use case 3: replace across entire DataFrame (all columns at once)
df_test = pd.DataFrame({"a": [1, 2, -1], "b": [-1, 3, -1]})
cleaned = df_test.replace(-1, np.nan)
print("Replace -1 sentinel with NaN across the whole DataFrame:")
print(cleaned)

Boolean replaced with Yes/No:
active
Yes    39
No     11
Name: count, dtype: int64

Replacing variant spellings:
department
Sales          13
Hr             13
Engineering    12
Finance        12
Name: count, dtype: int64

Replace -1 sentinel with NaN across the whole DataFrame:
     a    b
0  1.0  NaN
1  2.0  3.0
2  NaN  NaN


In [ ]:
# Side-by-side: .map() vs .replace() on the same data
# Demonstrates the key behavioural difference

small = pd.Series(["Engineering", "Sales", "HR", "Finance"])
lookup = {"Engineering": "Tech", "Sales": "Revenue"}

via_map     = small.map(lookup)
via_replace = small.replace(lookup)

result = pd.DataFrame({
    "original":    small,
    "via_map":     via_map,
    "via_replace": via_replace
})
print(result)
print()
print("  .map():     HR and Finance became NaN (not in the dict)")
print("  .replace(): HR and Finance stayed as-is (only specified values changed)")
print()
print("Use .map() when you want to transform EVERY value via a complete lookup.")
print("Use .replace() when you want to fix SPECIFIC values and leave the rest alone.")

---
### Exercise 2 — Map and Replace

**Task:** Use `.map()` and `.replace()` to clean and enrich the data.

In [ ]:
# Q1: Map department to a cost_centre code using the dictionary below
cost_centre_map = {
    "Engineering": "CC-101",
    "Sales":       "CC-102",
    "Hr":          "CC-103",
    "Finance":     "CC-104"
}
emp["cost_centre"] = emp['department'].map(cost_center_map)
print("Q1 Cost centre mapping:")
print(emp[["name", "department", "cost_centre"]].head(6))
missing_cc = emp["cost_centre"].isnull().sum()
print(f"Missing cost centres (unmatched departments): {missing_cc}")

# Q2: Use .replace() to fix the missing cost centres
#     The missing ones are likely 'Hr' vs 'HR' — check which department they are
print("\nDepartments with no cost centre:")
print(emp[emp["cost_centre"].isnull()]["department"].unique())

# Fix: add the correct mapping for whatever spelling appears above
emp["department"] = emp["department"].replace("Hr","HR")
cost_centre_map["HR"] = "CC-103
emp["cost_centre"] = emp["department"].map(cost_centre_map)
print(f"Missing cost centres after fix: {emp['cost_centre'].isnull().sum()}")

# Q3: Replace the boolean 'active' column with strings 'Active' / 'Inactive'
emp["status"] = emp["active"].map(True:"Active",False:"Inactive")
print("\nQ3 Status column:")
print(emp["status"].value_counts())

print()
print(f"Correct Q1 (some NaN expected): {emp['cost_centre'].notnull().sum() > 0}")
print(f"Correct Q3 (only Active/Inactive): {set(emp['status'].unique()) <= {'Active','Inactive'}}")

<details>
<summary>Show solution</summary>

```python
emp["cost_centre"] = emp["department"].map(cost_centre_map)

# The unmatched department spelling will be 'Hr' (title-cased from earlier cleaning)
emp["department"]  = emp["department"].replace("Hr", "HR")
# Update cost_centre_map to use 'HR' and remap
cost_centre_map["HR"] = "CC-103"
emp["cost_centre"] = emp["department"].map(cost_centre_map)

emp["status"] = emp["active"].replace({True: "Active", False: "Inactive"})
```

</details>

---
# Part 3 — Pivot and Melt

## Wide vs Long format — the fundamental concept

Data can be arranged in two fundamental shapes:

**Wide format** — each category gets its own column:
```
  employee    Q1_score   Q2_score   Q3_score   Q4_score
  Ananya      4.2        4.5        4.1        4.8
  Rohit       3.8        4.0        3.9        4.2
```

**Long format** — each observation gets its own row:
```
  employee    quarter    score
  Ananya      Q1         4.2
  Ananya      Q2         4.5
  Ananya      Q3         4.1
  Ananya      Q4         4.8
  Rohit       Q1         3.8
  ...         ...        ...
```

**Wide** is easy for humans to read and compare across categories. **Long** is what most analysis tools (including Pandas' groupby, and most plotting libraries) expect.

- `pd.pivot()` / `pd.pivot_table()` — Long → Wide
- `pd.melt()` — Wide → Long

In [ ]:
# Create a wide-format quarterly score dataset to work with

quarterly_scores = pd.DataFrame({
    "employee_id": [1001, 1002, 1003, 1004, 1005, 1006],
    "name":        ["Ananya", "Rohit", "Priya", "Vikram", "Sneha", "Arjun"],
    "department":  ["Engineering", "Sales", "HR", "Finance", "Engineering", "Sales"],
    "Q1_score":    [4.2, 3.8, 4.5, 3.9, 4.7, 3.5],
    "Q2_score":    [4.5, 4.0, 4.3, 4.1, 4.8, 3.8],
    "Q3_score":    [4.1, 3.9, 4.6, 4.0, 4.6, 4.0],
    "Q4_score":    [4.8, 4.2, 4.4, 3.8, 5.0, 4.1]
})

print("Wide format — quarterly_scores:")
print(quarterly_scores)

Wide format — quarterly_scores:
   employee_id    name   department  Q1_score  Q2_score  Q3_score  Q4_score
0         1001  Ananya  Engineering       4.2       4.5       4.1       4.8
1         1002   Rohit        Sales       3.8       4.0       3.9       4.2
2         1003   Priya           HR       4.5       4.3       4.6       4.4
3         1004  Vikram      Finance       3.9       4.1       4.0       3.8
4         1005   Sneha  Engineering       4.7       4.8       4.6       5.0
5         1006   Arjun        Sales       3.5       3.8       4.0       4.1


In [ ]:
# pd.melt() — Wide → Long
# id_vars:    columns to KEEP as identifiers (they repeat for each melted row)
# value_vars: columns to UNPIVOT into rows (the wide columns being melted down)
# var_name:   what to call the new column that holds the old column names
# value_name: what to call the new column that holds the values

long_scores = pd.melt(
    quarterly_scores,
    id_vars    = ["employee_id", "name", "department"],
    value_vars = ["Q1_score", "Q2_score", "Q3_score", "Q4_score"],
    var_name   = "quarter",
    value_name = "score"
)

print("Long format after melt:")
print(long_scores)
print()
print(f"Wide shape: {quarterly_scores.shape}   Long shape: {long_scores.shape}")
print(f"Rows went from {len(quarterly_scores)} to {len(long_scores)}")
print(f"(6 employees × 4 quarters = 24 rows)")

Long format after melt:
    employee_id    name   department   quarter  score
0          1001  Ananya  Engineering  Q1_score    4.2
1          1002   Rohit        Sales  Q1_score    3.8
2          1003   Priya           HR  Q1_score    4.5
3          1004  Vikram      Finance  Q1_score    3.9
4          1005   Sneha  Engineering  Q1_score    4.7
5          1006   Arjun        Sales  Q1_score    3.5
6          1001  Ananya  Engineering  Q2_score    4.5
7          1002   Rohit        Sales  Q2_score    4.0
8          1003   Priya           HR  Q2_score    4.3
9          1004  Vikram      Finance  Q2_score    4.1
10         1005   Sneha  Engineering  Q2_score    4.8
11         1006   Arjun        Sales  Q2_score    3.8
12         1001  Ananya  Engineering  Q3_score    4.1
13         1002   Rohit        Sales  Q3_score    3.9
14         1003   Priya           HR  Q3_score    4.6
15         1004  Vikram      Finance  Q3_score    4.0
16         1005   Sneha  Engineering  Q3_score    4.6
17  

In [ ]:
# The payoff: long format unlocks groupby analysis across quarters
# This would be impossible on the wide format without a loop

# Clean up the quarter label (remove '_score' suffix)
long_scores["quarter"] = long_scores["quarter"].str.replace("_score", "", regex=False)

# Average score per quarter
print("Average score per quarter:")
print(long_scores.groupby("quarter")["score"].mean().round(2))
print()

# Average score per department
print("Average score per department:")
print(long_scores.groupby("department")["score"].mean().round(2))
print()

# Average score per department AND quarter
print("Average score per department and quarter:")
print(long_scores.groupby(["department", "quarter"])["score"].mean().round(2).unstack())

In [ ]:
# pd.pivot() — Long → Wide (the reverse of melt)
# index:   which column becomes the row labels
# columns: which column's VALUES become the new column headers
# values:  which column holds the cell values

wide_again = long_scores.pivot(
    index   = "name",
    columns = "quarter",
    values  = "score"
)

print("Long → Wide again with pivot():")
print(wide_again)
print()
print("The column axis name 'quarter' can be removed for a cleaner look:")
wide_again.columns.name = None
print(wide_again)

Long → Wide again with pivot():
quarter  Q1_score  Q2_score  Q3_score  Q4_score
name                                           
Ananya        4.2       4.5       4.1       4.8
Arjun         3.5       3.8       4.0       4.1
Priya         4.5       4.3       4.6       4.4
Rohit         3.8       4.0       3.9       4.2
Sneha         4.7       4.8       4.6       5.0
Vikram        3.9       4.1       4.0       3.8

The column axis name 'quarter' can be removed for a cleaner look:
        Q1_score  Q2_score  Q3_score  Q4_score
name                                          
Ananya       4.2       4.5       4.1       4.8
Arjun        3.5       3.8       4.0       4.1
Priya        4.5       4.3       4.6       4.4
Rohit        3.8       4.0       3.9       4.2
Sneha        4.7       4.8       4.6       5.0
Vikram       3.9       4.1       4.0       3.8


### `pivot()` vs `pivot_table()` — the key difference

- `pd.pivot()` — requires the combination of `index` + `columns` to be **unique per cell**. If there are duplicate combinations, it raises an error.
- `pd.pivot_table()` — handles duplicates by **aggregating** them (mean, sum, count, etc.). Covered in Session 6.3.

Use `pivot()` on clean, already-unique data (like our quarterly_scores where each person has exactly one score per quarter). Use `pivot_table()` when you have multiple values per cell and want to aggregate them.

---
### Exercise 3 — Pivot and Melt

**Task:** Reshape the data between wide and long formats.

In [ ]:
# A wide-format budget table: one column per year, one row per department
budget_wide = pd.DataFrame({
    "department": ["Engineering", "Sales", "HR", "Finance"],
    "budget_2021": [7500000, 3800000, 1600000, 2800000],
    "budget_2022": [8000000, 4000000, 1700000, 3000000],
    "budget_2023": [8500000, 4200000, 1800000, 3100000]
})

print("Original wide-format budget table:")
print(budget_wide)
print()

# Q1: Melt budget_wide into long format
#     id_vars: department column
#     value_vars: the three budget columns
#     var_name: 'year_col'  |  value_name: 'budget'
budget_long = pd.melt(
    budget_wide,
    id_vars    = ['depratment'],
    value_vars = ['budget_2021',"budgest_2022","budget_2023"],
    var_name   = "year_col",
    value_name = "budget"
)
print("Q1 Melted (long format):")
print(budget_long)

# Q2: Clean the year column — extract just the 4-digit year from 'budget_2021' etc.
#     Hint: .str.split('_').str[-1] splits on underscore and takes the last part
budget_long["year"] = budget_long["year_col"].str.split("_").str[-1]
print("\nQ2 With clean year column:")
print(budget_long[["department", "year", "budget"]])

# Q3: Pivot the clean long format BACK to wide format
#     rows = department, columns = year, values = budget
budget_wide_again = budget_long[["department", "year", "budget"]].pivot(
    index   = 'department',
    columns = "year",
    values  ="values"
)
budget_wide_again.columns.name = None
print("\nQ3 Pivoted back to wide:")
print(budget_wide_again)

print()
print(f"Correct Q1 (12 rows = 4 depts x 3 years): {len(budget_long) == 12}")
print(f"Correct Q3 (shape 4x3): {budget_wide_again.shape == (4, 3)}")

<details>
<summary>Show solution</summary>

```python
budget_long = pd.melt(
    budget_wide,
    id_vars    = ["department"],
    value_vars = ["budget_2021", "budget_2022", "budget_2023"],
    var_name   = "year_col",
    value_name = "budget"
)

budget_long["year"] = budget_long["year_col"].str.split("_").str[-1]

budget_wide_again = budget_long[["department", "year", "budget"]].pivot(
    index="department", columns="year", values="budget"
)
```

</details>

In [ ]:
emp.head(10)

,employee_id,name,department,salary,experience,city,performance_rating,join_date,active,salary_band,salary_formatted,composite_score,region
0,1001,Vikram Reddy,Engineering,72000.0,19,Chennai,NaN,2011-03-01,True,Mid,"₹72,000",NaN,Chennai
1,1002,Tarun Joshi,Sales,68000.0,1,Delhi,3.8,2021-04-01,True,Mid,"₹68,000",42.4,North
2,1003,Priya Joshi,Hr,57000.0,20,Bangalore,4.9,2014-02-01,False,Entry,"₹57,000",74.6,South
3,1004,Rahul Kumar,Engineering,NaN,8,Bangalore,3.1,2021-05-01,True,Unknown,N/A,NaN,South
4,1005,Rahul Das,Engineering,100000.0,3,Hyderabad,3.4,2015-03-01,True,Senior,"₹100,000",51.6,South
5,1006,Lakshmi Sharma,Sales,NaN,11,Chennai,2.8,2021-04-01,True,Unknown,N/A,NaN,Chennai
6,1007,Meera Nair,Hr,60000.0,9,Hyderabad,3.5,2018-03-01,False,Mid,"₹60,000",50.5,South
7,1008,Priya Sharma,Hr,60000.0,6,Chennai,5.0,2020-08-01,False,Mid,"₹60,000",55.0,Chennai
8,1009,Ananya Reddy,Finance,104000.0,18,Bangalore,3.4,2012-09-01,True,Senior,"₹104,000",75.1,South
9,1010,Meera Bose,Engineering,93000.0,4,Bangalore,2.5,2004-12-01,True,Senior,"₹93,000",45.8,South


In [ ]:
emp.pivot_table(
    index='city',
    columns='department',
    values='experience',
    aggfunc='sum'
)

department,Engineering,Finance,Hr,Sales
city,,,,
Bangalore,30.0,18.0,53.0,NaN
Chennai,41.0,77.0,25.0,26.0
Delhi,34.0,31.0,28.0,49.0
Hyderabad,12.0,NaN,35.0,47.0
Mumbai,32.0,15.0,11.0,37.0


In [ ]:
emp.pivot(
    index='employee_id',
    columns='department',
    values='experience'
)

department,Engineering,Finance,Hr,Sales
employee_id,,,,
1001,19.0,NaN,NaN,NaN
1002,NaN,NaN,NaN,1.0
1003,NaN,NaN,20.0,NaN
1004,8.0,NaN,NaN,NaN
1005,3.0,NaN,NaN,NaN
1006,NaN,NaN,NaN,11.0
1007,NaN,NaN,9.0,NaN
1008,NaN,NaN,6.0,NaN
1009,NaN,18.0,NaN,NaN


---
# Part 4 — Date/Time Handling

## Why DateTime deserves its own section

Timestamps look like strings but behave like numbers — you can sort them, subtract them to get durations, and extract components like month, weekday, or hour. When Pandas knows a column is a datetime (dtype `datetime64`), it unlocks a whole toolkit via the `.dt` accessor.

We switch to the **sales transactions dataset** for this section because it has 300 transactions spread across a full year with exact timestamps — far richer for DateTime analysis than the monthly join_dates in the employee table.

## Step 0: Make sure Pandas knows the column IS a datetime

In [ ]:
print("sales dtypes:")
print(sales.dtypes)
print()
print(sales[["transaction_id", "timestamp", "product", "total_amount"]].head(5))
print()
print("'timestamp' dtype is:", sales["timestamp"].dtype)
print("It is datetime64 — we can use the .dt accessor.")
print()
print("If it were stored as a plain string (object), we would convert it first:")
print("  pd.to_datetime(df['col'])  <- converts a string column to datetime64")

sales dtypes:
transaction_id            object
timestamp         datetime64[ns]
product                   object
category                  object
region                    object
quantity                   int64
unit_price               float64
total_amount             float64
dtype: object

  transaction_id           timestamp   product  total_amount
0         TX0001 2023-02-07 16:06:00  Keyboard       14800.0
1         TX0002 2023-02-14 14:26:00  Keyboard       16000.0
2         TX0003 2023-01-31 17:07:00    Laptop       66000.0
3         TX0004 2023-01-26 11:02:00   Monitor      113000.0
4         TX0005 2023-10-04 09:36:00   Headset       13200.0

'timestamp' dtype is: datetime64[ns]
It is datetime64 — we can use the .dt accessor.

If it were stored as a plain string (object), we would convert it first:
  pd.to_datetime(df['col'])  <- converts a string column to datetime64


In [ ]:
pd.to_datetime

In [ ]:
import time

time.time()

1783349656.717248

In [ ]:
# The .dt accessor — extracts components from a datetime Series

sales["year"]        = sales["timestamp"].dt.year
sales["month"]       = sales["timestamp"].dt.month
sales["month_name"]  = sales["timestamp"].dt.month_name()
sales["day"]         = sales["timestamp"].dt.day
sales["weekday"]     = sales["timestamp"].dt.day_name()    # Monday, Tuesday...
sales["hour"]        = sales["timestamp"].dt.hour
sales["quarter"]     = sales["timestamp"].dt.quarter
sales["week"]        = sales["timestamp"].dt.isocalendar().week

print("Extracted datetime components:")
print(sales[["timestamp", "year", "month", "month_name", "weekday", "hour", "quarter"]].head(8))

Extracted datetime components:
            timestamp  year  month month_name    weekday  hour  quarter
0 2023-02-07 16:06:00  2023      2   February    Tuesday    16        1
1 2023-02-14 14:26:00  2023      2   February    Tuesday    14        1
2 2023-01-31 17:07:00  2023      1    January    Tuesday    17        1
3 2023-01-26 11:02:00  2023      1    January   Thursday    11        1
4 2023-10-04 09:36:00  2023     10    October  Wednesday     9        4
5 2023-11-24 11:23:00  2023     11   November     Friday    11        4
6 2023-04-16 15:43:00  2023      4      April     Sunday    15        2
7 2023-08-21 13:19:00  2023      8     August     Monday    13        3


In [ ]:
# Now we can do time-based analysis that was impossible without these components

# Revenue by month
print("Monthly revenue:")
monthly_rev = sales.groupby("month_name")["total_amount"].sum().round(0)
# Sort by month number, not alphabetically
month_order = sales.groupby("month")["month_name"].first().sort_index().values
print(monthly_rev.reindex(month_order))
print()

# Busiest day of the week
print("Revenue by day of week:")
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_rev   = sales.groupby("weekday")["total_amount"].sum().round(0)
print(day_rev.reindex([d for d in day_order if d in day_rev.index]))
print()

# Peak selling hour
print("Revenue by hour of day:")
print(sales.groupby("hour")["total_amount"].sum().round(0).sort_index())

In [ ]:
# DateTime arithmetic — subtracting timestamps to get durations
# Subtracting two datetime columns gives a Timedelta (duration)

# How many days has each transaction been since the start of 2023?
start_of_year = pd.Timestamp("2023-01-01")

sales["days_since_start"] = (sales["timestamp"] - start_of_year).dt.days

print("Days since start of 2023 for first 5 transactions:")
print(sales[["transaction_id", "timestamp", "days_since_start"]].head(5))
print()

# Employee example: how long has each employee been with the company?
reference_date = pd.Timestamp("2024-01-01")
emp["tenure_days"]  = (reference_date - pd.to_datetime(emp["join_date"])).dt.days
emp["tenure_years"] = (emp["tenure_days"] / 365.25).round(1)

print("Employee tenure:")
print(emp[["name", "join_date", "tenure_days", "tenure_years"]].head(5))

In [ ]:
# Filtering by date — works naturally with comparison operators
# because datetime columns are comparable

# Transactions in Q2 only (April, May, June)
q2_sales = sales[sales["quarter"] == 2]
print(f"Q2 transactions: {len(q2_sales)}")
print(f"Q2 total revenue: {q2_sales['total_amount'].sum():,.0f}")
print()

# Transactions between specific dates
start = pd.Timestamp("2023-06-01")
end   = pd.Timestamp("2023-08-31")
summer = sales[(sales["timestamp"] >= start) & (sales["timestamp"] <= end)]
print(f"June-August transactions: {len(summer)}")
print()

# Weekend transactions only
weekends = sales[sales["weekday"].isin(["Saturday", "Sunday"])]
print(f"Weekend transactions: {len(weekends)}  ({len(weekends)/len(sales)*100:.1f}% of total)")

In [ ]:
# .resample() — group by a time frequency automatically
# This requires the datetime column to be the index
# Think of it as groupby for time periods

sales_indexed = sales.set_index("timestamp")

# Total revenue per month ('ME' = month end)
monthly = sales_indexed["total_amount"].resample("ME").sum().round(0)
print("Monthly revenue via resample:")
print(monthly)
print()

# Total revenue per quarter
quarterly = sales_indexed["total_amount"].resample("QE").sum().round(0)
print("Quarterly revenue:")
print(quarterly)

---
### Exercise 4 — DateTime Handling

**Task:** Use the `sales` DataFrame to answer the time-based questions below.

In [ ]:
# Q1: Extract the hour from the timestamp and store it in a column called 'hour'
#     (it may already exist — recreate it to practice the syntax)
sales["hour"] = sales["timestamp"].___.___()
print(f"Q1 Hour range: {sales['hour'].min()} to {sales['hour'].max()}")

# Q2: How many transactions happened in each quarter?
q2 = sales.groupby(___)["transaction_id"].count()
print("\nQ2 Transactions per quarter:")
print(q2)

# Q3: Find the day (from the timestamp) with the single highest total revenue
#     Hint: extract the date part first with .dt.date, then groupby and find the max
sales["date"] = sales["timestamp"].dt.___
daily_rev     = sales.groupby(___)["total_amount"].sum()
best_day      = daily_rev.___()
print(f"\nQ3 Best revenue day: {best_day}  (revenue: {daily_rev[best_day]:,.0f})")

# Q4: Filter sales to transactions that happened in the AFTERNOON (hour >= 12)
#     and compute the average transaction value for that subset
afternoon = sales[sales[___] >= ___]
avg_afternoon = afternoon["total_amount"].___()
print(f"\nQ4 Afternoon transactions: {len(afternoon)}")
print(f"    Average transaction value: {avg_afternoon:,.0f}")

# Q5: Which MONTH had the highest total revenue? Return the month name.
monthly_totals = sales.groupby(___)["total_amount"].sum()
best_month_num = monthly_totals.___()
best_month_name = sales[sales["month"] == best_month_num]["month_name"].iloc[0]
print(f"\nQ5 Best revenue month: {best_month_name}  (month number: {best_month_num})")

print()
print(f"Correct Q2 (4 quarters):        {len(q2) == 4}")
print(f"Correct Q4 (fewer than total):  {len(afternoon) < len(sales)}")

<details>
<summary>Show solution</summary>

```python
sales["hour"]  = sales["timestamp"].dt.hour

q2 = sales.groupby("quarter")["transaction_id"].count()

sales["date"] = sales["timestamp"].dt.date
daily_rev     = sales.groupby("date")["total_amount"].sum()
best_day      = daily_rev.idxmax()

afternoon     = sales[sales["hour"] >= 12]
avg_afternoon = afternoon["total_amount"].mean()

monthly_totals  = sales.groupby("month")["total_amount"].sum()
best_month_num  = monthly_totals.idxmax()
```

</details>

---
## Final Exercise — End-to-End Challenge

The management team wants a comprehensive Q4 performance report combining employee data and sales data. Complete all four steps — each uses a different tool from today.

In [ ]:
# Step 1 — Apply: Create a 'performance_tier' column on the employees DataFrame
# Rules (use the composite_score column created in Part 1):
#   composite_score >= 75  -> 'High'
#   composite_score >= 50  -> 'Medium'
#   anything else or NaN   -> 'Low'

def performance_tier(score):
    if pd.isna(score):
        return ___
    elif score >= ___:
        return ___
    elif score >= ___:
        return ___
    return ___

emp["performance_tier"] = emp["composite_score"].apply(___)

print("Step 1 — Performance tier distribution:")
print(emp["performance_tier"].value_counts())

In [ ]:
# Step 2 — Map: Add a 'region' column to employees using the city_to_region dict
#               (region was already added in Part 2, but let's rebuild it cleanly)

city_to_region = {
    "Mumbai": "West", "Delhi": "North",
    "Bangalore": "South", "Chennai": "South", "Hyderabad": "South"
}

emp["region"] = emp[___].map(___)

print("Step 2 — Employee count per region:")
print(emp["region"].value_counts())

In [ ]:
# Step 3 — Pivot: Build a wide-format headcount table
#                 Rows = region, Columns = performance_tier, Values = headcount

headcount_long = emp.groupby(["region", "performance_tier"])["employee_id"].count().reset_index()
headcount_long.columns = ["region", "performance_tier", "headcount"]

headcount_wide = headcount_long.pivot(
    index   = ___,
    columns = ___,
    values  = ___
).fillna(0).astype(int)
headcount_wide.columns.name = None

print("Step 3 — Headcount by region and performance tier:")
print(headcount_wide)

In [ ]:
# Step 4 — DateTime: From the sales data, compute Q4 (Oct-Dec) revenue per region
#           Then merge it with the employee headcount_wide table
#           to see revenue per employee headcount per region

# Filter sales to Q4
q4_sales = sales[sales[___] == 4]
print(f"Q4 transactions: {len(q4_sales)}")

# Revenue per region in Q4
q4_revenue = q4_sales.groupby("region")["total_amount"].sum().round(0).reset_index()
q4_revenue.columns = ["region", "q4_revenue"]
print("\nQ4 revenue per region:")
print(q4_revenue)

# Add total headcount column to headcount_wide for the merge
hw = headcount_wide.copy().reset_index()
hw["total_headcount"] = hw[[c for c in hw.columns if c != "region"]].sum(axis=1)

# Merge Q4 revenue with employee headcount
final_report = pd.merge(hw, q4_revenue, on=___, how=___)

# Revenue per employee head
final_report["revenue_per_head"] = (final_report[___] / final_report[___]).round(0)

print("\nFINAL REPORT — Q4 Revenue vs Headcount by Region")
print("=" * 55)
print(final_report[["region", "total_headcount", "q4_revenue", "revenue_per_head"]])

print()
print(f"Correct (Q4 = quarter 4):           {q4_sales['quarter'].unique().tolist() == [4]}")
print(f"Correct (final report has 3 rows):  {len(final_report) >= 2}")

<details>
<summary>Show solutions for all four steps</summary>

```python
# Step 1
def performance_tier(score):
    if pd.isna(score):   return "Low"
    elif score >= 75:    return "High"
    elif score >= 50:    return "Medium"
    return "Low"
emp["performance_tier"] = emp["composite_score"].apply(performance_tier)

# Step 2
emp["region"] = emp["city"].map(city_to_region)

# Step 3
headcount_wide = headcount_long.pivot(
    index="region", columns="performance_tier", values="headcount"
).fillna(0).astype(int)

# Step 4
q4_sales   = sales[sales["quarter"] == 4]
final_report = pd.merge(hw, q4_revenue, on="region", how="left")
final_report["revenue_per_head"] = (final_report["q4_revenue"] / final_report["total_headcount"]).round(0)
```

</details>

---
## Session Summary

### Apply functions
- `series.apply(func)` — runs `func` on every value in a column
- `df.apply(func, axis=1)` — runs `func` on every row (receives row as a Series, access by column name)
- `lambda x: expression` — one-liner anonymous function for simple transformations
- Use `.apply()` only when no vectorized alternative exists — it is much slower on large data
- Always handle `pd.isna()` inside the function if the column has missing values

### Map and Replace
- `.map(dict)` — translates every value via a lookup dict; values not in the dict become `NaN`
- `.replace(old, new)` — substitutes specific values, leaves everything else unchanged
- `.replace({k: v, ...})` — substitute multiple values in one call
- After `.map()`, always check `.isnull().sum()` to catch missing mappings
- Key difference: `.map()` transforms every value; `.replace()` only touches specified ones

### Pivot and Melt
- **Wide** = one column per category (readable by humans, hard for groupby)
- **Long** = one row per observation (what groupby/plotting tools expect)
- `pd.melt(df, id_vars, value_vars, var_name, value_name)` — Wide → Long
- `df.pivot(index, columns, values)` — Long → Wide (fails if index+columns are not unique)
- `pd.pivot_table()` — Long → Wide with aggregation for duplicate combinations

### DateTime handling
- `pd.to_datetime(series)` — converts a string column to `datetime64`
- `.dt.year`, `.dt.month`, `.dt.day`, `.dt.hour`, `.dt.weekday`, `.dt.quarter` — extract components
- `.dt.month_name()`, `.dt.day_name()` — human-readable names
- Subtracting two datetime columns gives a `Timedelta`; use `.dt.days` to get an integer
- Filter by date using normal comparison operators: `df[df['ts'] >= pd.Timestamp('2023-06-01')]`
- `.resample('ME')` — group by time frequency (requires datetime as the index)

---

### What comes next

You now have the complete Pandas transformation toolkit. The next sessions move into **data visualisation** — taking the clean, transformed DataFrames you have been building and turning them into charts that communicate insights to a non-technical audience.

---

### Quick reference

```python
# Apply
df['col'].apply(func)             # per value in a column
df.apply(func, axis=1)            # per row
df['col'].apply(lambda x: x * 2) # lambda shorthand

# Map and Replace
df['col'].map({'A': 1, 'B': 2})          # full lookup, unmatched -> NaN
df['col'].replace('old', 'new')           # specific substitution
df['col'].replace(['a','b'], 'c')         # list of values to one replacement
df.replace({'col1': {0: np.nan}})         # targeted column replacement

# Melt (Wide -> Long)
pd.melt(df, id_vars=['id'], value_vars=['c1','c2'],
        var_name='category', value_name='value')

# Pivot (Long -> Wide)
df.pivot(index='row_col', columns='col_col', values='val_col')

# DateTime
pd.to_datetime(df['col'])         # parse strings to datetime
df['ts'].dt.year                  # extract year
df['ts'].dt.month_name()          # 'January', 'February'...
df['ts'].dt.day_name()            # 'Monday', 'Tuesday'...
df['ts'].dt.hour                  # 0-23
df['ts'].dt.quarter               # 1-4
df['ts'].dt.date                  # date only, no time
(df['ts2'] - df['ts1']).dt.days   # duration in days
df.set_index('ts').resample('ME').sum()  # monthly totals
```